# Stage 2B — APS Shi–Tomasi vs FAST comparison

This experiment compares Shi–Tomasi and FAST on identical consecutive DAVIS APS frame pairs. Both use an 8 × 6 spatial grid, 5 px minimum selected-feature spacing, and requested caps of 50–1,000. The FAST threshold is set to 1 for this measured run so FAST supplies enough candidates to compare the requested operating points; this is an intentionally low threshold and should be treated as a tunable parameter. Measurements include selected count, occupied spatial buckets, native detector response, raw-coordinate mutual-nearest repeatability within 3 px, and detection-plus-selection runtime.

Native response scales are detector-specific and are not directly comparable. Repeatability is not tracking accuracy: no image-motion compensation, optical flow, or KLT is used. This stage does not implement event association or event-driven tracking.

In [ ]:
import importlib
import sys
from pathlib import Path

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "detector_comparison.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.davis_inspection import DavisConfig, load_davis
import src.frame_synchronization as frame_synchronization
frame_synchronization = importlib.reload(frame_synchronization)
import src.detector_comparison as detector_comparison
detector_comparison = importlib.reload(detector_comparison)

config = DavisConfig()
recording = load_davis(config)
origins_s, _ = frame_synchronization.load_timestamp_origins(recording, config)
windows = frame_synchronization.make_frame_windows(recording, origins_s)
detector_config = detector_comparison.DetectorConfig(
    fast_threshold=1,
    timing_repeats=3,
)
feature_rows, repeatability_rows, parameters, frame_lookup = (
    detector_comparison.compare_detectors(windows, config=detector_config)
)
outputs = detector_comparison.save_comparison_outputs(
    feature_rows,
    repeatability_rows,
    parameters,
    frame_lookup,
    detector_comparison.DEFAULT_OUTPUT_DIR,
)
for name, path in outputs.items():
    print(f"{name}: {path}")

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(outputs["report"].read_text()))
for key in ("overlays", "spatial_distribution", "metrics_plot"):
    display(Image(filename=str(outputs[key])))